<a href="https://colab.research.google.com/github/Z-C95/Data-Engineering-UTN/blob/main/BerrazLeandro_TP1!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP1 - Data Engineering: Extracción y almacenamiento de datos
**Instituto:** UTN BA - Centro de E-Learning  
**Materia:** Data Engineering  
**Módulo:** Nº1 - Extracción y almacenamiento de datos

---

## Fuente de datos: Alpha Vantage API

Se utilizaron **2 endpoints** de la misma API sobre acciones argentinas que cotizan en NYSE como ADRs:

| Endpoint | Datos | Tipo de extracción |
|---|---|---|
| `TIME_SERIES_DAILY` | Precio de cierre, apertura, máx, mín, volumen diario | **Incremental** (append + partición por fecha) |
| `OVERVIEW` | Sector, industria, CEO, descripción, país | **Full** (overwrite) |

### Justificación de decisiones
- **API elegida:** Alpha Vantage por ser gratuita, usar `requests` estándar, y tener endpoints claramente diferenciados entre datos temporales y metadatos estáticos.
- **Acciones elegidas:** YPF, Pampa Energía (PAM) y Vista Energy (VIST), tres empresas del sector energético argentino que cotizan en NYSE.
- **Extracción incremental** para precios diarios: cada ejecución agrega solo los registros nuevos (fechas no presentes en la tabla), particionando por `year/month/day`.
- **Extracción full** para OVERVIEW: los metadatos de empresa cambian raramente → siempre se sobreescribe con la versión más actual.
- **Delta Lake** con la librería `deltalake` (sin Spark), compatible con Google Colab.
- **Rate limiting:** se agrega una pausa de 15 segundos entre cada request para respetar el límite de 5 requests/minuto del plan gratuito.


## 0. Instalación de dependencias

In [ ]:
import subprocess, sys

def instalar(paquete):
    subprocess.check_call([sys.executable, "-m", "pip", "install", paquete, "-q"])

instalar("deltalake")
instalar("requests")
instalar("pandas")

print("Dependencias instaladas correctamente.")

Dependencias instaladas correctamente.


## 1. Imports y configuración general

In [ ]:
import requests
import pandas as pd
import time
from deltalake import DeltaTable, write_deltalake
from datetime import datetime, timezone
import os

# ── Configuración de la API
API_KEY        = "RYT4BCOGLYD0RR8C"
BASE_URL       = "https://www.alphavantage.co/query"
SLEEP_SECONDS  = 15   # pausa entre requests para respetar el límite de 5/min del plan free

# Tickers de las tres empresas argentinas (ADRs en NYSE)
SYMBOLS = ["YPF", "PAM", "VIST"]

# Rutas del data lake
PATH_PRICES   = "data_lake/raw/prices"    # datos temporales → incremental
PATH_OVERVIEW = "data_lake/raw/overview"  # metadatos        → full

print("Configuración lista.")
print(f"  Symbols        : {SYMBOLS}")
print(f"  Prices         → {PATH_PRICES}")
print(f"  Overview       → {PATH_OVERVIEW}")
print(f"  Pausa entre requests: {SLEEP_SECONDS}s")

Configuración lista.
  Symbols        : ['YPF', 'PAM', 'VIST']
  Prices         → data_lake/raw/prices
  Overview       → data_lake/raw/overview
  Pausa entre requests: 15s


## 2. Funciones auxiliares

In [ ]:
def get_json(params: dict, descripcion: str = "") -> dict:
    """
    Realiza una petición GET a la API con pausa automática post-request
    para respetar el rate limit del plan gratuito (5 requests/minuto).
    """
    params["apikey"] = API_KEY
    print(f"  → Request: {descripcion or params.get('function', '')} ... ", end="", flush=True)
    response = requests.get(BASE_URL, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()
    print("OK")
    print(f"     Esperando {SLEEP_SECONDS}s para respetar rate limit...", flush=True)
    time.sleep(SLEEP_SECONDS)
    return data


def asegurar_directorio(path: str):
    """Crea el directorio si no existe."""
    os.makedirs(path, exist_ok=True)
    print(f"  Directorio listo: {path}")


def tabla_existe(path: str) -> bool:
    """Verifica si ya existe una tabla Delta en el path dado."""
    return os.path.exists(os.path.join(path, "_delta_log"))

## 3. Extracción incremental: Precios diarios (datos temporales)

Se consulta `TIME_SERIES_DAILY` para cada símbolo.  
La extracción es **incremental**: solo se agregan fechas que aún no están en la tabla Delta.  
Se particiona por `year / month / day`.


In [ ]:
def extraer_prices_symbol(symbol: str) -> pd.DataFrame:
    """Obtiene la serie de precios diarios de un símbolo. Devuelve un DataFrame."""
    data = get_json(
        {"function": "TIME_SERIES_DAILY", "symbol": symbol, "outputsize": "compact"},
        descripcion=f"TIME_SERIES_DAILY {symbol}"
    )

    serie = data.get("Time Series (Daily)", {})
    if not serie:
        raise ValueError(f"Sin datos para {symbol}. Respuesta: {data}")

    registros = []
    for fecha_str, valores in serie.items():
        fecha = datetime.strptime(fecha_str, "%Y-%m-%d")
        registros.append({
            "symbol":  symbol,
            "date":    fecha_str,
            "open":    float(valores["1. open"]),
            "high":    float(valores["2. high"]),
            "low":     float(valores["3. low"]),
            "close":   float(valores["4. close"]),
            "volume":  int(valores["5. volume"]),
            "year":    fecha.year,
            "month":   fecha.month,
            "day":     fecha.day,
        })

    df = pd.DataFrame(registros)
    df["extraction_ts"] = datetime.now(timezone.utc).isoformat()
    return df


def extraer_prices_incremental() -> pd.DataFrame:
    """
    Extrae precios de todos los símbolos y filtra solo los registros nuevos
    (fechas que aún no están almacenadas en la tabla Delta).
    """
    print("[PRICES] Extrayendo precios diarios...")
    frames = []
    for symbol in SYMBOLS:
        df = extraer_prices_symbol(symbol)
        frames.append(df)

    df_nuevo = pd.concat(frames, ignore_index=True)

    # Filtrado incremental: descartar fechas ya almacenadas
    if tabla_existe(PATH_PRICES):
        df_existente = DeltaTable(PATH_PRICES).to_pandas()[["symbol", "date"]]
        fechas_existentes = set(zip(df_existente["symbol"], df_existente["date"]))
        mask = ~df_nuevo.apply(lambda r: (r["symbol"], r["date"]) in fechas_existentes, axis=1)
        df_nuevo = df_nuevo[mask].reset_index(drop=True)
        print(f"  Registros nuevos a agregar: {len(df_nuevo)}")
    else:
        print(f"  Primera carga. Registros a insertar: {len(df_nuevo)}")

    return df_nuevo


def guardar_prices(df: pd.DataFrame):
    """Guarda en Delta Lake en modo APPEND, particionado por year/month/day."""
    if df.empty:
        print("  Sin registros nuevos. Nada que guardar.")
        return

    asegurar_directorio(PATH_PRICES)
    write_deltalake(
        PATH_PRICES,
        df,
        mode="append",
        partition_by=["year", "month", "day"],
    )
    print(f"  [OK] Prices guardados en: {PATH_PRICES}")


# Ejecución
df_prices = extraer_prices_incremental()
if not df_prices.empty:
    print()
    display(df_prices[["symbol", "date", "open", "close", "volume"]]
            .groupby("symbol").head(3)
            .sort_values(["symbol", "date"], ascending=[True, False]))
print()
guardar_prices(df_prices)

[PRICES] Extrayendo precios diarios...
  → Request: TIME_SERIES_DAILY YPF ... OK
     Esperando 15s para respetar rate limit...
  → Request: TIME_SERIES_DAILY PAM ... OK
     Esperando 15s para respetar rate limit...
  → Request: TIME_SERIES_DAILY VIST ... OK
     Esperando 15s para respetar rate limit...
  Registros nuevos a agregar: 0

  Sin registros nuevos. Nada que guardar.


## 4. Extracción full: Overview de empresas (datos estáticos)

Se consulta `OVERVIEW` para cada símbolo.  
Se guarda en modo **overwrite**: siempre refleja el estado actual de los metadatos.


In [ ]:
def extraer_overview() -> pd.DataFrame:
    """Obtiene metadatos de cada empresa. Datos estáticos/relativamente estables."""
    print("[OVERVIEW] Extrayendo metadatos de empresas...")
    columnas = [
        "Symbol", "Name", "Exchange", "Currency", "Country",
        "Sector", "Industry", "Description", "FullTimeEmployees",
        "FiscalYearEnd", "LatestQuarter", "MarketCapitalization",
        "PERatio", "EPS", "DividendYield", "52WeekHigh", "52WeekLow",
        "AnalystTargetPrice", "Address"
    ]

    registros = []
    for symbol in SYMBOLS:
        data = get_json(
            {"function": "OVERVIEW", "symbol": symbol},
            descripcion=f"OVERVIEW {symbol}"
        )
        if not data or "Symbol" not in data:
            print(f"  Advertencia: sin datos de overview para {symbol}")
            continue
        registro = {col: data.get(col, None) for col in columnas}
        registros.append(registro)

    df = pd.DataFrame(registros)
    df.columns = [c.lower() for c in df.columns]
    df["extraction_ts"] = datetime.now(timezone.utc).isoformat()

    print(f"  Registros obtenidos: {len(df)}")
    return df


def guardar_overview(df: pd.DataFrame):
    """Guarda en Delta Lake en modo OVERWRITE (full). Sin partición."""
    asegurar_directorio(PATH_OVERVIEW)

    # Castear todo a string para evitar columnas con tipo Null puro
    df = df.astype(str)

    write_deltalake(
        PATH_OVERVIEW,
        df,
        mode="overwrite",
    )
    print(f"  [OK] Overview guardado en: {PATH_OVERVIEW}")


# Ejecución
df_overview = extraer_overview()
print()
display(df_overview[["symbol", "name", "sector", "industry", "country"]])
print()
guardar_overview(df_overview)

[OVERVIEW] Extrayendo metadatos de empresas...
  → Request: OVERVIEW YPF ... OK
     Esperando 15s para respetar rate limit...
  → Request: OVERVIEW PAM ... OK
     Esperando 15s para respetar rate limit...
  Advertencia: sin datos de overview para PAM
  → Request: OVERVIEW VIST ... OK
     Esperando 15s para respetar rate limit...
  Registros obtenidos: 2



,symbol,name,sector,industry,country
0,YPF,YPF Sociedad Anonima,ENERGY,OIL & GAS INTEGRATED,USA
1,VIST,Vista Oil Gas ADR,ENERGY,OIL & GAS E&P,USA



  Directorio listo: data_lake/raw/overview
  [OK] Overview guardado en: data_lake/raw/overview


## 5. Verificación: lectura desde Delta Lake

Se leen ambas tablas para confirmar que los datos fueron almacenados correctamente.


In [ ]:
print("=" * 55)
print("  VERIFICACIÓN - Lectura desde Delta Lake")
print("=" * 55)

# Tabla prices
dt_prices = DeltaTable(PATH_PRICES)
df_p = dt_prices.to_pandas()
print(f"\nTabla PRICES → {len(df_p)} registros acumulados")
display(df_p[["symbol", "date", "open", "close", "volume"]].sort_values("date", ascending=False).head(9))

# Tabla overview
dt_overview = DeltaTable(PATH_OVERVIEW)
df_o = dt_overview.to_pandas()
print(f"\nTabla OVERVIEW → {len(df_o)} registros")
display(df_o[["symbol", "name", "sector", "industry", "marketcapitalization"]])

print("\n  Extracción y almacenamiento completados exitosamente.")

  VERIFICACIÓN - Lectura desde Delta Lake

Tabla PRICES → 300 registros acumulados


,symbol,date,open,close,volume
260,PAM,2026-06-02,87.52,85.73,194178
259,VIST,2026-06-02,76.28,76.72,419213
258,YPF,2026-06-02,54.54,55.31,1800053
201,VIST,2026-06-01,76.09,76.02,882246
202,YPF,2026-06-01,53.90,54.74,2934131
203,PAM,2026-06-01,85.92,87.62,367104
122,PAM,2026-05-29,84.09,85.25,339351
121,YPF,2026-05-29,52.18,53.01,1893636
120,VIST,2026-05-29,73.00,74.20,846887



Tabla OVERVIEW → 2 registros


,symbol,name,sector,industry,marketcapitalization
0,YPF,YPF Sociedad Anonima,ENERGY,OIL & GAS INTEGRATED,23096791000
1,VIST,Vista Oil Gas ADR,ENERGY,OIL & GAS E&P,8471851000



  Extracción y almacenamiento completados exitosamente.
